# 🌍 TerraLens AI — LULC com Random Forest

**Análise:** Classificação de Uso e Cobertura do Solo

**Pipeline:**
1. Download de imagens Sentinel-2 via Google Earth Engine
2. Extração de features espectrais
3. Criação de amostras de treino/validação
4. Treino do Random Forest
5. Avaliação de métricas
6. Exportação do modelo para o backend

**Datasets:** MapBiomas, Sentinel-2, PRODES/INPE

In [ ]:
# ── INSTALAÇÃO DAS DEPENDÊNCIAS ─────────────────────────────────────
# !pip install scikit-learn numpy pillow rasterio geopandas matplotlib earthengine-api

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pickle
import os

print('✅ Dependências carregadas!')

## 1. Configuração e Classes

In [ ]:
# Classes de uso do solo (ajuste para sua área de estudo)
CLASSES = {
    0: 'Vegetação Densa',
    1: 'Agricultura',
    2: 'Solo Exposto',
    3: 'Corpo dAgua',
    4: 'Área Urbana',
    5: 'Floresta',
    6: 'Pastagem',
    7: 'Área Úmida',
}

N_CLASSES = len(CLASSES)
print(f'Total de classes: {N_CLASSES}')
for k, v in CLASSES.items():
    print(f'  {k}: {v}')

## 2. Extração de Features

In [ ]:
def extract_features(image_path: str) -> np.ndarray:
    """
    Extrai features espectrais de uma imagem RGB.
    Para Sentinel-2 real: use bandas B2, B3, B4, B8 (NIR) via rasterio.
    """
    img = Image.open(image_path).convert('RGB')
    img = img.resize((256, 256))
    arr = np.array(img, dtype=np.float32) / 255.0

    R = arr[:, :, 0]
    G = arr[:, :, 1]
    B = arr[:, :, 2]

    # Índices espectrais (proxies RGB)
    NGRDI = (G - R) / (G + R + 1e-9)           # NDVI proxy
    NDWI  = (G - B) / (G + B + 1e-9)           # Água
    GLI   = (2*G - R - B) / (2*G + R + B + 1e-9)  # Vegetação
    ExG   = 2*G - R - B                         # Excesso de verde
    VARI  = (G - R) / (G + R - B + 1e-9)        # VARI

    # Stack de features: 8 canais por pixel
    features = np.stack([R, G, B, NGRDI, NDWI, GLI, ExG, VARI], axis=-1)
    return features.reshape(-1, 8)


def extract_features_sentinel2(tif_path: str) -> np.ndarray:
    """
    Extrai features de imagem Sentinel-2 multibanda (GeoTIFF).
    Requer rasterio e imagem com bandas: B2(Blue), B3(Green), B4(Red), B8(NIR)
    """
    import rasterio
    with rasterio.open(tif_path) as src:
        B2 = src.read(1).astype(np.float32) / 10000.0  # Blue
        B3 = src.read(2).astype(np.float32) / 10000.0  # Green
        B4 = src.read(3).astype(np.float32) / 10000.0  # Red
        B8 = src.read(4).astype(np.float32) / 10000.0  # NIR

    # NDVI real com NIR
    NDVI  = (B8 - B4) / (B8 + B4 + 1e-9)
    NDWI  = (B3 - B8) / (B3 + B8 + 1e-9)
    EVI   = 2.5 * (B8 - B4) / (B8 + 6*B4 - 7.5*B2 + 1)

    features = np.stack([B2, B3, B4, B8, NDVI, NDWI, EVI], axis=-1)
    H, W, _ = features.shape
    return features.reshape(-1, 7)


print('✅ Funções de extração de features prontas!')

## 3. Criação de Amostras de Treino

In [ ]:
# OPÇÃO A: Gerar amostras sintéticas para testar o pipeline
# Substitua por amostras reais coletadas via QGIS ou GEE

def generate_synthetic_samples(n_per_class: int = 200) -> tuple:
    """
    Gera amostras sintéticas para teste do pipeline.
    Em produção: carregue amostras reais de CSV ou shapefile.
    """
    np.random.seed(42)
    X_list, y_list = [], []

    # Vegetação Densa (NGRDI alto, NDWI baixo)
    for _ in range(n_per_class):
        r, g, b = np.random.uniform(0.05, 0.2), np.random.uniform(0.3, 0.6), np.random.uniform(0.05, 0.2)
        X_list.append(make_features(r, g, b))
        y_list.append(0)

    # Água (B alto, G médio)
    for _ in range(n_per_class):
        r, g, b = np.random.uniform(0.02, 0.1), np.random.uniform(0.1, 0.3), np.random.uniform(0.3, 0.7)
        X_list.append(make_features(r, g, b))
        y_list.append(3)

    # Solo Exposto (R alto, G médio, B baixo)
    for _ in range(n_per_class):
        r, g, b = np.random.uniform(0.4, 0.8), np.random.uniform(0.3, 0.6), np.random.uniform(0.1, 0.3)
        X_list.append(make_features(r, g, b))
        y_list.append(2)

    # Urbano (valores altos e similares nos 3 canais)
    for _ in range(n_per_class):
        v = np.random.uniform(0.4, 0.7)
        r, g, b = v + np.random.normal(0, 0.05), v + np.random.normal(0, 0.05), v + np.random.normal(0, 0.05)
        X_list.append(make_features(np.clip(r,0,1), np.clip(g,0,1), np.clip(b,0,1)))
        y_list.append(4)

    return np.array(X_list), np.array(y_list)


def make_features(r, g, b):
    ngrdi = (g - r) / (g + r + 1e-9)
    ndwi  = (g - b) / (g + b + 1e-9)
    gli   = (2*g - r - b) / (2*g + r + b + 1e-9)
    exg   = 2*g - r - b
    vari  = (g - r) / (g + r - b + 1e-9)
    return [r, g, b, ngrdi, ndwi, gli, exg, vari]


X, y = generate_synthetic_samples(300)
print(f'✅ Amostras geradas: {X.shape[0]} pixels, {X.shape[1]} features')
print(f'   Classes: {np.unique(y)}')

## 4. Treino e Avaliação

In [ ]:
# Split treino/validação
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Treino do Random Forest
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

print('🚀 Treinando Random Forest...')
clf.fit(X_train, y_train)

# Avaliação
y_pred = clf.predict(X_val)
acc = accuracy_score(y_val, y_pred)

print(f'\n✅ Acurácia Overall: {acc:.4f} ({acc*100:.1f}%)')
print('\nRelatório por classe:')
print(classification_report(y_val, y_pred, target_names=[CLASSES[c] for c in sorted(CLASSES.keys()) if c in np.unique(y)]))

## 5. Importância das Features

In [ ]:
feature_names = ['R', 'G', 'B', 'NGRDI', 'NDWI', 'GLI', 'ExG', 'VARI']
importances = clf.feature_importances_

plt.figure(figsize=(10, 4))
plt.bar(feature_names, importances, color='#1e7fff')
plt.title('Importância das Features — Random Forest LULC')
plt.ylabel('Importância')
plt.xlabel('Feature')
plt.tight_layout()
plt.savefig('feature_importance_lulc.png', dpi=150)
plt.show()
print('✅ Gráfico salvo!')

## 6. Exportar Modelo para o Backend

In [ ]:
# Salva o modelo treinado
output_path = '../../backend/models/sat/lulc/random_forest_lulc.pkl'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'wb') as f:
    pickle.dump(clf, f)

print(f'✅ Modelo salvo em: {output_path}')
print(f'   Acurácia: {acc*100:.1f}%')
print(f'   Features: {feature_names}')
print(f'   Classes: {list(CLASSES.values())}')
print(f'\n🚀 Agora o TerraLens AI vai rodar o modelo REAL!')